In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / 'data').exists():
    raise FileNotFoundError('Não foi possível localizar a raiz do projeto.')

for candidate in [
    PROJECT_ROOT / 'code',
    PROJECT_ROOT / 'code' / 'revenue',
    PROJECT_ROOT / 'code' / 'tmdb',
]:
    if candidate.exists() and str(candidate.resolve()) not in sys.path:
        sys.path.append(str(candidate.resolve()))

import pandas as pd

from imbalance_experiment_utils import (
    HYBRID_CLASSIFIER_CONFIGS,
    ORACLE_BAND_ARTIFACT_DIR,
    REVENUE_BAND_LABELS,
    SOFT_ROUTING_CLASSIFICATION_REGRESSION_ARTIFACT_DIR,
    TMDB_EXTENDED_NO_TRANSFORM_ARTIFACT_DIR,
    load_best_params_lookup_from_artifact_dir,
    load_results_from_artifact_dir,
    load_tmdb_extended_context,
    run_soft_routing_classification_regression_experiment,
    save_additional_table,
    save_artifact_tables,
    summarize_errors_by_band,
)


/home/gabriel/Faculdade/Matérias/ML/UFSJ_Aprendizado_Maquina_TP1/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# **Abordagem Híbrida com Soft Routing na Base TMDB Estendida**

Este notebook repete a lógica de classificação + regressão do notebook `08`, mas substitui o roteamento duro por um roteamento suave. Em vez de escolher um único regressor de faixa, a previsão final combina os especialistas ponderando suas saídas pelas probabilidades previstas pelo classificador.

O objetivo é verificar se o ganho potencial mostrado pelo oracle do notebook `07` pode ser parcialmente recuperado sem exigir acerto exato da faixa em todos os casos.

In [2]:
TARGET_NAME = 'Sem transformação'
OVERWRITE_ARTIFACTS = False
SHOW_PROGRESS = True
MIN_BAND_SAMPLES = 80

ARTIFACT_DIR = SOFT_ROUTING_CLASSIFICATION_REGRESSION_ARTIFACT_DIR
ERROR_ANALYSIS_DIR = ARTIFACT_DIR / 'error_analysis'
BEST_COMPARISON_PATH = ARTIFACT_DIR / 'best_soft_routing_vs_global_and_oracle.csv'
CONFUSION_PATH = ERROR_ANALYSIS_DIR / '09_soft_routing_confusion_matrix.csv'
BAND_METRICS_PATH = ERROR_ANALYSIS_DIR / '09_soft_routing_metricas_por_faixa.csv'
TRAINING_SIZES_PATH = ARTIFACT_DIR / 'training_band_sizes.csv'
CONFIDENCE_SUMMARY_PATH = ARTIFACT_DIR / 'classifier_confidence_summary.csv'


In [3]:
context = load_tmdb_extended_context()
df_movies = context['df_movies']
X = context['X']
y = context['y']
folds_df = context['folds_df']
revenue_bins = context['revenue_bins']

print(f'Base TMDB estendida: {df_movies.shape[0]} filmes | {X.shape[1]} features')
display(df_movies[['id_tmdb', 'title', 'revenue']].head())


Base TMDB estendida: 6918 filmes | 284 features


,id_tmdb,title,revenue
0,552524,Lilo & Stitch,610800000
1,950387,A Minecraft Movie,947000000
2,1257960,सिकंदर,24727058
3,574475,Final Destination Bloodlines,229314062
4,1197306,A Working Man,98652557


In [4]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
ERROR_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

if (
    not OVERWRITE_ARTIFACTS
    and (ARTIFACT_DIR / 'model_selection_results.csv').exists()
    and (ARTIFACT_DIR / 'model_selection_predictions.csv').exists()
    and (ARTIFACT_DIR / 'model_selection_summary.csv').exists()
    and TRAINING_SIZES_PATH.exists()
):
    results_df, predictions_df, summary_df = load_results_from_artifact_dir(ARTIFACT_DIR)
    training_sizes_df = pd.read_csv(TRAINING_SIZES_PATH)
else:
    best_params_lookup = load_best_params_lookup_from_artifact_dir(
        TMDB_EXTENDED_NO_TRANSFORM_ARTIFACT_DIR,
        target_name=TARGET_NAME,
    )
    results_df, predictions_df, summary_df, training_sizes_df = run_soft_routing_classification_regression_experiment(
        df_movies=df_movies,
        X=X,
        y=y,
        folds_df=folds_df,
        revenue_bins=revenue_bins,
        best_params_lookup=best_params_lookup,
        classifier_configs=HYBRID_CLASSIFIER_CONFIGS,
        target_name=TARGET_NAME,
        min_band_samples=MIN_BAND_SAMPLES,
        show_progress=SHOW_PROGRESS,
    )
    save_artifact_tables(
        ARTIFACT_DIR,
        results_df=results_df,
        predictions_df=predictions_df,
        summary_df=summary_df,
    )
    training_sizes_df.to_csv(TRAINING_SIZES_PATH, index=False)

summary_df = summary_df.loc[summary_df['target_version'] == TARGET_NAME].copy()
global_results_df, global_predictions_df, global_summary_df = load_results_from_artifact_dir(
    TMDB_EXTENDED_NO_TRANSFORM_ARTIFACT_DIR,
)
oracle_results_df, oracle_predictions_df, oracle_summary_df = load_results_from_artifact_dir(
    ORACLE_BAND_ARTIFACT_DIR,
)
global_summary_df = global_summary_df.loc[global_summary_df['target_version'] == TARGET_NAME].copy()
oracle_summary_df = oracle_summary_df.loc[oracle_summary_df['target_version'] == TARGET_NAME].copy()

best_soft_row = summary_df.sort_values(['mean_rmse', 'mean_mae', 'model']).iloc[0]
best_global_row = global_summary_df.sort_values(['mean_rmse', 'mean_mae', 'model']).iloc[0]
best_oracle_row = oracle_summary_df.sort_values(['mean_rmse', 'mean_mae', 'model']).iloc[0]

best_comparison_df = pd.DataFrame([
    {
        'cenário': 'Melhor soft routing',
        'modelo': best_soft_row['model'],
        'mean_rmse': best_soft_row['mean_rmse'],
        'mean_mae': best_soft_row['mean_mae'],
        'mean_r2': best_soft_row['mean_r2'],
        'mean_band_accuracy': best_soft_row['mean_band_accuracy'],
        'mean_band_macro_f1': best_soft_row['mean_band_macro_f1'],
    },
    {
        'cenário': 'Melhor global TMDB estendido',
        'modelo': best_global_row['model'],
        'mean_rmse': best_global_row['mean_rmse'],
        'mean_mae': best_global_row['mean_mae'],
        'mean_r2': best_global_row['mean_r2'],
        'mean_band_accuracy': None,
        'mean_band_macro_f1': None,
    },
    {
        'cenário': 'Melhor oracle por faixa',
        'modelo': best_oracle_row['model'],
        'mean_rmse': best_oracle_row['mean_rmse'],
        'mean_mae': best_oracle_row['mean_mae'],
        'mean_r2': best_oracle_row['mean_r2'],
        'mean_band_accuracy': None,
        'mean_band_macro_f1': None,
    },
])
best_comparison_df['delta_rmse_vs_global'] = best_comparison_df['mean_rmse'] - best_global_row['mean_rmse']
best_comparison_df['delta_mae_vs_global'] = best_comparison_df['mean_mae'] - best_global_row['mean_mae']
best_comparison_df['delta_r2_vs_global'] = best_comparison_df['mean_r2'] - best_global_row['mean_r2']
best_comparison_df['delta_rmse_vs_oracle'] = best_comparison_df['mean_rmse'] - best_oracle_row['mean_rmse']
best_comparison_df['delta_mae_vs_oracle'] = best_comparison_df['mean_mae'] - best_oracle_row['mean_mae']
best_comparison_df['delta_r2_vs_oracle'] = best_comparison_df['mean_r2'] - best_oracle_row['mean_r2']
save_additional_table(best_comparison_df, BEST_COMPARISON_PATH)

best_soft_predictions_df = predictions_df.loc[predictions_df['model'] == best_soft_row['model']].copy()
band_metrics_df = summarize_errors_by_band(best_soft_predictions_df, revenue_bins)
save_additional_table(band_metrics_df, BAND_METRICS_PATH)

confusion_df = pd.crosstab(
    pd.Categorical(best_soft_predictions_df['true_band'], categories=REVENUE_BAND_LABELS, ordered=True),
    pd.Categorical(best_soft_predictions_df['predicted_band'], categories=REVENUE_BAND_LABELS, ordered=True),
    normalize='index',
)
confusion_df = confusion_df.reindex(index=REVENUE_BAND_LABELS, columns=REVENUE_BAND_LABELS, fill_value=0.0)
confusion_df.index.name = 'true_band'
save_additional_table(confusion_df.reset_index(), CONFUSION_PATH)

band_code_lookup = {label: index for index, label in enumerate(REVENUE_BAND_LABELS)}
true_codes = best_soft_predictions_df['true_band'].map(band_code_lookup).to_numpy(dtype=int)
predicted_codes = best_soft_predictions_df['predicted_band'].map(band_code_lookup).to_numpy(dtype=int)
confidence_summary_df = pd.DataFrame([
    {
        'model': best_soft_row['model'],
        'mean_confidence': best_soft_predictions_df['classifier_confidence'].mean(),
        'median_confidence': best_soft_predictions_df['classifier_confidence'].median(),
        'p25_confidence': best_soft_predictions_df['classifier_confidence'].quantile(0.25),
        'p75_confidence': best_soft_predictions_df['classifier_confidence'].quantile(0.75),
        'min_confidence': best_soft_predictions_df['classifier_confidence'].min(),
        'max_confidence': best_soft_predictions_df['classifier_confidence'].max(),
        'band_accuracy': (true_codes == predicted_codes).mean(),
        'band_accuracy_pm1': (abs(true_codes - predicted_codes) <= 1).mean(),
    }
])
save_additional_table(confidence_summary_df, CONFIDENCE_SUMMARY_PATH)

display(summary_df[[
    'model',
    'classifier_model',
    'regressor_model',
    'mean_rmse',
    'mean_mae',
    'mean_r2',
    'mean_band_accuracy',
    'mean_band_macro_f1',
]])
display(best_comparison_df)
display(band_metrics_df)
display(confusion_df)
display(confidence_summary_df)


Soft routing: 100%|██████████| 590/590 [1:35:34<00:00,  9.72s/ajuste, XGBoost Classifier | fold 9 | melhor=0.4960 | refit=sim]            


,model,classifier_model,regressor_model,mean_rmse,mean_mae,mean_r2,mean_band_accuracy,mean_band_macro_f1
0,Gradient Boosting Classifier + Gradient Boosti...,Gradient Boosting Classifier,Gradient Boosting Regressor,1.086528e+08,5.409844e+07,0.650025,0.477449,0.472852
1,XGBoost Classifier + XGBoost Regressor,XGBoost Classifier,XGBoost Regressor,1.108808e+08,5.422416e+07,0.634246,0.468341,0.463101
2,Random Forest Classifier + Random Forest Regre...,Random Forest Classifier,Random Forest Regressor,1.154180e+08,5.792771e+07,0.608695,0.457646,0.442782


,cenário,modelo,mean_rmse,mean_mae,mean_r2,mean_band_accuracy,mean_band_macro_f1,delta_rmse_vs_global,delta_mae_vs_global,delta_r2_vs_global,delta_rmse_vs_oracle,delta_mae_vs_oracle,delta_r2_vs_oracle
0,Melhor soft routing,Gradient Boosting Classifier + Gradient Boosti...,1.086528e+08,5.409844e+07,0.650025,0.477449,0.472852,-3.012075e+06,-1.059068e+06,0.019849,1.179830e+07,2.074033e+07,-0.069690
1,Melhor global TMDB estendido,XGBoost Regressor,1.116649e+08,5.515751e+07,0.630175,NaN,NaN,0.000000e+00,0.000000e+00,0.000000,1.481038e+07,2.179940e+07,-0.089539
2,Melhor oracle por faixa,Gradient Boosting Regressor,9.685453e+07,3.335812e+07,0.719715,NaN,NaN,-1.481038e+07,-2.179940e+07,0.089539,0.000000e+00,0.000000e+00,0.000000


,faixa_receita,quantidade_filmes,receita_minima,receita_maxima,mae_medio,residuo_medio,mediana_erro_absoluto,percentual_subestimados,percentual_superestimados,rmse
0,Muito baixa receita,1384,1,7086000,2.331526e+07,-2.331169e+07,1.737415e+07,0.361272,99.638728,3.684385e+07
1,Baixa receita,1383,7096000,22441323,2.870540e+07,-2.802230e+07,2.166631e+07,8.749096,91.250904,4.432403e+07
2,Média receita,1384,22468044,52800000,2.803021e+07,-2.162435e+07,1.715318e+07,27.745665,72.254335,4.575000e+07
3,Alta receita,1383,52900000,134038006,3.911412e+07,-1.106744e+06,2.864465e+07,61.243673,38.756327,5.684244e+07
4,Muito alta receita,1384,134100000,2923706026,1.512994e+08,7.979586e+07,1.074499e+08,72.832370,27.167630,2.262598e+08


col_0,Muito baixa receita,Baixa receita,Média receita,Alta receita,Muito alta receita
true_band,,,,,
Muito baixa receita,0.651734,0.192197,0.097543,0.049855,0.008671
Baixa receita,0.289949,0.331164,0.205351,0.151121,0.022415
Média receita,0.136561,0.233382,0.263728,0.306358,0.059971
Alta receita,0.052784,0.106291,0.199566,0.456255,0.185105
Muito alta receita,0.012283,0.036850,0.065751,0.200867,0.684249


,model,mean_confidence,median_confidence,p25_confidence,p75_confidence,min_confidence,max_confidence,band_accuracy,band_accuracy_pm1
0,Gradient Boosting Classifier + Gradient Boosti...,0.484201,0.416464,0.346728,0.562873,0.224566,0.997012,0.47745,0.839983
